In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('../data/raw/pizza_sales.csv')
df.head()

,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1,1,hawaiian_m,1,1/1/2015,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2,2,classic_dlx_m,1,1/1/2015,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3,2,five_cheese_l,1,1/1/2015,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4,2,ital_supr_l,1,1/1/2015,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5,2,mexicana_m,1,1/1/2015,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


# Data Cleaning

In [3]:
df.isnull().sum()

pizza_id             0
order_id             0
pizza_name_id        0
quantity             0
order_date           0
order_time           0
unit_price           0
total_price          0
pizza_size           0
pizza_category       0
pizza_ingredients    0
pizza_name           0
dtype: int64

In [4]:
duplicate_data=df[df.duplicated()]
duplicate_data

,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name


# Feature Engineering

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pizza_id           48620 non-null  int64  
 1   order_id           48620 non-null  int64  
 2   pizza_name_id      48620 non-null  object 
 3   quantity           48620 non-null  int64  
 4   order_date         48620 non-null  object 
 5   order_time         48620 non-null  object 
 6   unit_price         48620 non-null  float64
 7   total_price        48620 non-null  float64
 8   pizza_size         48620 non-null  object 
 9   pizza_category     48620 non-null  object 
 10  pizza_ingredients  48620 non-null  object 
 11  pizza_name         48620 non-null  object 
dtypes: float64(2), int64(3), object(7)
memory usage: 4.5+ MB


## Functions

In [6]:
def dateConverter(dataframe, *args):
    for col in args:
        dataframe[col]=pd.to_datetime(dataframe[col],format='mixed')
    print('Datetime converted')
    
def currencyConverter(dataframe,prefix,*args):
    for col in args:
        dataframe[col]=dataframe[col].map(f'{prefix}''{:,.2f}'.format)
        
def pctConverter(dataframe,suffix,*args):
    for col in args:
        dataframe[col] = dataframe[col].map('{:,.2f}'f'{suffix}'.format)

In [7]:
dateConverter(df,'order_date','order_time')

Datetime converted


In [8]:
# Day, Month, Hour

In [9]:
df['Order_Day']=df['order_date'].dt.day_name()
df['Order_Month']=df['order_date'].dt.month_name()
df['month_num']=df['order_date'].dt.month
df['Order_Hour']=df['order_time'].dt.hour

In [10]:
df['Order_Day'].unique()

array(['Thursday', 'Sunday', 'Wednesday', 'Friday', 'Monday', 'Saturday',
       'Tuesday'], dtype=object)

# KPI

In [11]:
# Total Revenue = Sum of total_price
# Total Pizzas Sold = Sum of quantity
# Total Orders = Count of unique order_id
# Average Order Value (AOV) = Total Revenue ÷ Total Orders
# Average Pizza per Order = Total Pizzas Sold ÷ Total Orders

In [12]:
total_revenue= df['total_price'].sum()
total_pizza_sold=df['quantity'].sum()
total_orders=df['order_id'].nunique()
average_order_value=total_revenue/total_orders
average_pizza_per_order = total_pizza_sold/total_orders

In [13]:
print(f'Total Revenue: ${total_revenue:,.2f}')
print(f'Total Pizza Sold (Quantity): {total_pizza_sold} qty')
print(f'Total Orders: {total_orders}')
print(f'Average Order Value: ${average_order_value:,.2f}')
print(f'Average Pizza Per Order: {average_pizza_per_order:,.2f}')

Total Revenue: $817,860.05
Total Pizza Sold (Quantity): 49574 qty
Total Orders: 21350
Average Order Value: $38.31
Average Pizza Per Order: 2.32


# Analysis

## Sales by day of the week

In [14]:
weekdays=['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']
df['Order_Day']=pd.Categorical(df['Order_Day'],categories=weekdays)

In [15]:
day_sales=df.groupby('Order_Day',observed=False)['total_price'].sum().reset_index().rename(
    columns={'Order_Day':"Weekdays",'total_price':"Total Price"}
)
currencyConverter(day_sales,'$','Total Price')
day_sales

,Weekdays,Total Price
0,Sunday,"$102,116.45"
1,Monday,"$110,471.60"
2,Tuesday,"$115,594.45"
3,Wednesday,"$116,731.20"
4,Thursday,"$121,650.30"
5,Friday,"$129,690.90"
6,Saturday,"$121,605.15"


## Sales by Hour of Day

In [16]:
hour_sales=df.groupby('Order_Hour')['total_price'].sum().reset_index().rename(
    columns={'Order_Hour':'Order Hour','total_price':'Total Price'}
)
currencyConverter(hour_sales,'$','Total Price')
hour_sales

,Order Hour,Total Price
0,9,$83.00
1,10,$303.65
2,11,"$44,935.80"
3,12,"$111,877.90"
4,13,"$106,065.70"
5,14,"$59,201.40"
6,15,"$52,992.30"
7,16,"$70,055.40"
8,17,"$86,237.45"
9,18,"$89,296.85"


## Monthly Sales

In [17]:
monthly_sales=df.groupby(['month_num','Order_Month']).agg(
    Total_Price =('total_price','sum'),
    Total_Orders = ('order_id','nunique')
).reset_index()[['Order_Month','Total_Price','Total_Orders']]

currencyConverter(monthly_sales,'$','Total_Price')

monthly_sales.columns=monthly_sales.columns.str.replace('_',' ',regex=False)
monthly_sales

,Order Month,Total Price,Total Orders
0,January,"$71,620.15",1929
1,February,"$64,419.45",1648
2,March,"$71,301.40",1864
3,April,"$70,312.00",1829
4,May,"$67,648.80",1765
5,June,"$68,161.45",1771
6,July,"$70,880.65",1860
7,August,"$69,497.30",1835
8,September,"$63,803.70",1638
9,October,"$68,152.20",1782


## % Change of Sales by Category

In [18]:
cat_sales_pct=df.groupby('pizza_category')['total_price'].sum().reset_index().rename(
    columns={'pizza_category':'Pizza Category','total_price':'Total Price'}
)
cat_sales_pct['Sales%']=(cat_sales_pct['Total Price']/total_revenue)*100
currencyConverter(cat_sales_pct,'$','Total Price')
pctConverter(cat_sales_pct,'%','Sales%')
cat_sales_pct

,Pizza Category,Total Price,Sales%
0,Chicken,"$195,919.50",23.96%
1,Classic,"$220,053.10",26.91%
2,Supreme,"$208,197.00",25.46%
3,Veggie,"$193,690.45",23.68%


## % Sales by Pizza Size and Category

In [19]:
size_cat_pct = df.groupby(['pizza_size','pizza_category'])['total_price'].sum().reset_index().rename(
    columns={'pizza_size':'Pizza Size','pizza_category':'Pizza Category','total_price':'Total Price'}
)
size_cat_pct['Sales%']=(size_cat_pct['Total Price']/total_revenue)*100
size_cat_pct = size_cat_pct.sort_values(ascending=True, by='Sales%',ignore_index=True)
currencyConverter(size_cat_pct,'$','Total Price')
pctConverter(size_cat_pct,'%','Sales%')
size_cat_pct

,Pizza Size,Pizza Category,Total Price,Sales%
0,XXL,Classic,"$1,006.60",0.12%
1,XL,Classic,"$14,076.00",1.72%
2,S,Chicken,"$28,356.00",3.47%
3,S,Veggie,"$32,386.75",3.96%
4,S,Supreme,"$47,463.50",5.80%
5,M,Veggie,"$57,101.00",6.98%
6,M,Classic,"$60,581.75",7.41%
7,M,Chicken,"$65,224.50",7.98%
8,M,Supreme,"$66,475.00",8.13%
9,S,Classic,"$69,870.25",8.54%


## Total Pizza Sold by Pizza Category

In [20]:
total_pizza=df.groupby('pizza_category')['quantity'].sum().reset_index().rename(
    columns={'pizza_category':'Pizza Category','quantity':'Quantity'}
).sort_values(ascending=False,by='Quantity',ignore_index=True)
total_pizza

,Pizza Category,Quantity
0,Classic,14888
1,Supreme,11987
2,Veggie,11649
3,Chicken,11050


## Top 5 Best Selling Pizza

In [21]:
top_5=df.groupby('pizza_name').agg(
    Total_Revenue=('total_price','sum'),
    Total_Orders=('order_id','nunique'),
    Total_Quantity=('quantity','sum')
).reset_index().sort_values(ascending=False,by='Total_Revenue',ignore_index=True).head()
currencyConverter(top_5,'$','Total_Revenue')
top_5.columns=top_5.columns.str.replace('_',' ',regex=False)
top_5

,pizza name,Total Revenue,Total Orders,Total Quantity
0,The Thai Chicken Pizza,"$43,434.25",2225,2371
1,The Barbecue Chicken Pizza,"$42,768.00",2273,2432
2,The California Chicken Pizza,"$41,409.50",2197,2370
3,The Classic Deluxe Pizza,"$38,180.50",2329,2453
4,The Spicy Italian Pizza,"$34,831.25",1822,1924


## 5 Least Selling Pizza

In [22]:
bottom_5=df.groupby('pizza_name').agg(
    Total_Revenue=('total_price','sum'),
    Total_Orders=('order_id','nunique'),
    Total_Quantity=('quantity','sum')
).reset_index().sort_values(ascending=True,by='Total_Revenue',ignore_index=True).head()
currencyConverter(bottom_5,'$','Total_Revenue')
bottom_5.columns=top_5.columns.str.replace('_',' ',regex=False)
bottom_5

,pizza name,Total Revenue,Total Orders,Total Quantity
0,The Brie Carre Pizza,"$11,588.50",480,490
1,The Green Garden Pizza,"$13,955.75",976,997
2,The Spinach Supreme Pizza,"$15,277.75",918,950
3,The Mediterranean Pizza,"$15,360.50",912,934
4,The Spinach Pesto Pizza,"$15,596.00",945,970
